In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, to_date
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType


:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-92f69425-9f1a-4163-9fcc-05bd18970fa5;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
:: resolution report :: resolve 151ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.sp

✅ Spark iniciado


In [ ]:
spark = SparkSession.builder \
    .appName("silver-layer") \
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark iniciado")

In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.silver")

DataFrame[]

In [3]:
df_bronze = spark.read.table("lakehouse.bronze.transacoes")

df_bronze.show(5)

+---+--------------------+--------------------+-------------+
| id|             payload|       data_ingestao|data_particao|
+---+--------------------+--------------------+-------------+
|  1|{"valor": 2321.91...|2026-03-29 23:31:...|   2026-03-29|
|  2|{"valor": 4130.44...|2026-03-29 23:31:...|   2026-03-29|
|  3|{"valor": 4633.68...|2026-03-29 23:31:...|   2026-03-29|
|  4|{"valor": 3685.2,...|2026-03-29 23:31:...|   2026-03-29|
|  5|{"valor": 1817.05...|2026-03-29 23:31:...|   2026-03-29|
+---+--------------------+--------------------+-------------+
only showing top 5 rows



In [4]:
df_silver = df_bronze \
    .dropna(subset=["transacao_id", "valor"]) \
    .withColumn("cliente_id", col("cliente_id").cast("int")) \
    .withColumn("valor", col("valor").cast("double")) \
    .withColumn("quantidade", col("quantidade").cast("int")) \
    .withColumn("data_transacao", to_timestamp("data_transacao")) \
    .dropDuplicates(["transacao_id"])

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `transacao_id` cannot be resolved. Did you mean one of the following? [`id`, `payload`, `data_ingestao`, `data_particao`].

In [ ]:
df_silver = df_silver.withColumn(
    "data_particao",
    to_date(col("data_transacao"))
)

In [ ]:
df_silver.writeTo("lakehouse.silver.transacoes") \
    .partitionedBy("data_particao") \
    .createOrReplace()

print("✅ Silver criada")

In [ ]:
df_silver.createOrReplaceTempView("updates")

spark.sql("""
MERGE INTO lakehouse.silver.transacoes t
USING updates u
ON t.transacao_id = u.transacao_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [ ]:
spark.read.table("lakehouse.silver.transacoes").show(5)